# 09 – Extra analyses requested by my supervisor
Neural models on the holdout, the scaling effect tested on its own for the neural models, and balanced accuracy for the benchmarks. Set `QUICK_TEST = False` for the full run.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import os, time, random
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             matthews_corrcoef, confusion_matrix)

from google.colab import drive
drive.mount('/content/drive')

BASE     = "/content/drive/MyDrive/CS5500_dissertation"
FEATURES = f"{BASE}/features"
RESULTS  = f"{BASE}/results"

TICKERS      = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]
DEV_END      = "2023-12-31"
HOLDOUT_FROM = "2024-01-01"
SEQ_LEN      = 20

INITIAL_TRAIN, TEST_SIZE, STEP = 1000, 250, 250
EPOCHS, PATIENCE, BATCH_SIZE, LR, UNITS, DROPOUT = 30, 5, 64, 1e-3, 32, 0.2

gpus = tf.config.list_physical_devices('GPU')
print("GPU:", bool(gpus))

dev, hold = {}, {}
for t in TICKERS:
    f = pd.read_csv(f"{FEATURES}/{t}_features.csv", index_col=0, parse_dates=True)
    dev[t]  = f.loc[:DEV_END]
    hold[t] = f.loc[HOLDOUT_FROM:]

PREDICTORS = [c for c in dev["AAPL"].columns
              if not c.startswith("target") and c != "price_today"]

print(f"{len(PREDICTORS)} predictors; development {len(dev['AAPL'])} rows, "
      f"holdout {len(hold['AAPL'])} rows per asset")

In [ ]:
QUICK_TEST = True     # True: one asset, one seed. Set False for the full run.

SEEDS  = [42] if QUICK_TEST else [42, 43, 44, 45, 46]
ASSETS = ["AAPL"] if QUICK_TEST else TICKERS
N_SCALING_FOLDS = 2 if QUICK_TEST else 6

print(f"Mode: {'QUICK TEST' if QUICK_TEST else 'FULL RUN'}")
print(f"Assets: {ASSETS}")
print(f"Seeds:  {SEEDS}")
print(f"\nTask 1 runs: {len(ASSETS) * 3 * len(SEEDS)}")
print(f"Task 2 runs: {len(ASSETS) * 3 * len(SEEDS) * 2 * N_SCALING_FOLDS}")

## 2. Shared machinery

In [ ]:
def set_seeds(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)


def make_sequences(X, y, seq_len=SEQ_LEN):
    n = len(X) - seq_len + 1
    return np.stack([X[i:i + seq_len] for i in range(n)]), y[seq_len - 1:]


def build_lstm(shape):
    return keras.Sequential([layers.Input(shape=shape), layers.LSTM(UNITS),
                             layers.Dropout(DROPOUT),
                             layers.Dense(1, activation="sigmoid")], name="LSTM")

def build_gru(shape):
    return keras.Sequential([layers.Input(shape=shape), layers.GRU(UNITS),
                             layers.Dropout(DROPOUT),
                             layers.Dense(1, activation="sigmoid")], name="GRU")

def build_cnn_lstm(shape):
    return keras.Sequential([
        layers.Input(shape=shape),
        layers.Conv1D(32, 3, activation="relu", padding="causal"),
        layers.MaxPooling1D(2), layers.LSTM(UNITS), layers.Dropout(DROPOUT),
        layers.Dense(1, activation="sigmoid")], name="CNN-LSTM")


ARCHITECTURES = {"LSTM": build_lstm, "GRU": build_gru, "CNN-LSTM": build_cnn_lstm}


def train_predict(build_fn, Xtr, ytr, Xte, seed):
    set_seeds(seed)
    m = build_fn(Xtr.shape[1:])
    m.compile(optimizer=keras.optimizers.Adam(LR),
              loss="binary_crossentropy", metrics=["accuracy"])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                         restore_best_weights=True)
    m.fit(Xtr, ytr, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.15,
          callbacks=[stop], verbose=0)
    return (m.predict(Xte, verbose=0).ravel() > 0.5).astype(int)


def score_all(y_true, y_pred, **meta):
    """Accuracy, balanced accuracy and MCC with a 95% CI on accuracy."""
    acc = accuracy_score(y_true, y_pred)
    se  = np.sqrt(acc * (1 - acc) / len(y_true))
    return {**meta,
            "n":            len(y_true),
            "accuracy":     round(acc, 4),
            "ci_low":       round(acc - 1.96 * se, 4),
            "ci_high":      round(acc + 1.96 * se, 4),
            "balanced_acc": round(balanced_accuracy_score(y_true, y_pred), 4),
            "mcc":          round(matthews_corrcoef(y_true, y_pred), 4),
            "pred_up_rate": round(float(np.mean(y_pred)), 4)}


def get_folds(df, n_folds=None):
    folds, start = [], INITIAL_TRAIN
    while start + TEST_SIZE <= len(df):
        folds.append((start, start + TEST_SIZE)); start += STEP
    return folds[-n_folds:] if n_folds else folds


print("Machinery ready")

## 3. Task 1 — Neural models on the 2024–2025 holdout
Trained on all development data through 2023-12-31; the scaler is fitted on development data only.

In [ ]:
rows = []
t0 = time.time()

for ticker in ASSETS:
    d, h = dev[ticker], hold[ticker]

    # scaler fitted on development data only
    scaler = StandardScaler().fit(d[PREDICTORS].values)

    Xd = scaler.transform(d[PREDICTORS].values)
    yd = d["target_direction"].values

    # lead-in window from the end of development, inputs only
    lead   = d[PREDICTORS].values[-(SEQ_LEN - 1):]
    Xh_raw = np.vstack([lead, h[PREDICTORS].values])
    Xh     = scaler.transform(Xh_raw)
    yh     = np.concatenate([np.zeros(SEQ_LEN - 1, dtype=int),
                             h["target_direction"].values])

    Xtr, ytr = make_sequences(Xd, yd)
    Xte, yte = make_sequences(Xh, yh)

    assert len(Xte) == len(h), f"expected {len(h)} holdout sequences, got {len(Xte)}"

    # zero-parameter benchmarks on the same holdout labels
    y_true = h["target_direction"].values
    rows.append(score_all(y_true, np.ones(len(y_true), dtype=int),
                          ticker=ticker, model="majority_class"))
    rows.append(score_all(y_true, (h["ret_lag_1"].values > 0).astype(int),
                          ticker=ticker, model="persistence"))

    # logistic regression, for continuity with notebook 08
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(Xd, yd)
    rows.append(score_all(y_true, clf.predict(scaler.transform(h[PREDICTORS].values)),
                          ticker=ticker, model="logistic_regression"))

    for arch, fn in ARCHITECTURES.items():
        for seed in SEEDS:
            pred = train_predict(fn, Xtr, ytr, Xte, seed)
            rows.append(score_all(yte, pred, ticker=ticker,
                                  model=arch, seed=seed))
            print(f"  {ticker:<5} {arch:<9} seed {seed}  "
                  f"acc {rows[-1]['accuracy']:.4f}  "
                  f"bal {rows[-1]['balanced_acc']:.4f}  "
                  f"mcc {rows[-1]['mcc']:+.4f}  ({time.time()-t0:.0f}s)")

holdout_neural = pd.DataFrame(rows)
print(f"\nComplete in {time.time()-t0:.0f}s")

In [ ]:
print("Holdout performance, mean across assets and seeds\n")

summary_holdout = (holdout_neural
    .groupby("model")[["accuracy", "balanced_acc", "mcc", "pred_up_rate"]]
    .mean().round(4)
    .sort_values("accuracy", ascending=False))
print(summary_holdout.to_string())

maj = holdout_neural.query("model == 'majority_class'").set_index("ticker")["accuracy"]
per = holdout_neural.query("model == 'persistence'").set_index("ticker")["accuracy"]

print("\n\nNeural models against the zero-parameter benchmarks\n")
for arch in ARCHITECTURES:
    a = holdout_neural.query("model == @arch").groupby("ticker")["accuracy"].mean()
    print(f"  {arch:<9} beats majority on {int((a > maj[a.index]).sum())}/{len(a)}, "
          f"persistence on {int((a > per[a.index]).sum())}/{len(a)}")

suffix = "_quick" if QUICK_TEST else ""
holdout_neural.to_csv(f"{RESULTS}/holdout_neural{suffix}.csv", index=False)

## 4. Task 2 — Isolating the scaling factor for neural models
The only thing that changes between the two arms is the data used to fit the scaler.

In [ ]:
def run_scaling_arm(df, build_fn, seed, global_scaling, n_folds):
    """Walk-forward, direction target, purged folds. Only the scaler differs."""
    X = df[PREDICTORS].values
    y = df["target_direction"].values

    global_scaler = StandardScaler().fit(X) if global_scaling else None

    y_true, y_pred = [], []

    for split, end in get_folds(df, n_folds):
        sc  = global_scaler if global_scaling else StandardScaler().fit(X[:split])
        Xsc = sc.transform(X)

        Xtr, ytr = make_sequences(Xsc[:split], y[:split])
        Xte, yte = make_sequences(Xsc[split - SEQ_LEN + 1:end],
                                  y[split - SEQ_LEN + 1:end])

        y_pred.extend(train_predict(build_fn, Xtr, ytr, Xte, seed))
        y_true.extend(yte)

    return np.array(y_true), np.array(y_pred)


scaling_rows = []
t0 = time.time()
total = len(ASSETS) * len(ARCHITECTURES) * len(SEEDS) * 2
done = 0

for ticker in ASSETS:
    for arch, fn in ARCHITECTURES.items():
        for seed in SEEDS:
            for label, glob in [("fold_fitted", False), ("globally_fitted", True)]:
                yt, yp = run_scaling_arm(dev[ticker], fn, seed, glob, N_SCALING_FOLDS)
                scaling_rows.append(score_all(yt, yp, ticker=ticker,
                                              architecture=arch, seed=seed,
                                              scaling=label))
                done += 1
                print(f"[{done:>3}/{total}] {ticker:<5} {arch:<9} seed {seed} "
                      f"{label:<16} acc {scaling_rows[-1]['accuracy']:.4f} "
                      f"({time.time()-t0:.0f}s)")

scaling = pd.DataFrame(scaling_rows)
print(f"\nComplete in {time.time()-t0:.0f}s")
scaling.to_csv(f"{RESULTS}/scaling_neural{suffix}.csv", index=False)

In [ ]:
print("Scaling effect by architecture\n")

pv = scaling.pivot_table(index=["ticker", "architecture", "seed"],
                         columns="scaling", values="accuracy").reset_index()
pv["difference"] = pv["globally_fitted"] - pv["fold_fitted"]

eff = (pv.groupby("architecture")["difference"]
         .agg(mean_effect_pts=lambda s: round(100 * s.mean(), 3),
              sd_pts=lambda s: round(100 * s.std(), 3),
              n="size",
              n_nonzero=lambda s: int((s.abs() > 1e-9).sum())))
print(eff.to_string())

overall = 100 * pv["difference"].mean()
print(f"\nPooled scaling effect: {overall:+.3f} percentage points "
      f"across {len(pv)} matched pairs")
print(f"Pairs with any non-zero difference: "
      f"{int((pv['difference'].abs() > 1e-9).sum())}/{len(pv)}")

print("\nComparison with the linear result\n")
print("  Ordinary least squares:  0.000 pts (exact, all four configurations)")
print(f"  Neural architectures:   {overall:+.3f} pts (pooled)")
print("\nA non-zero neural effect alongside an exactly zero linear effect would")
print("show that leakage severity depends on the estimator rather than only on")
print("the presence of the leakage pathway.")

eff.to_csv(f"{RESULTS}/scaling_effect_by_architecture{suffix}.csv")

## 5. Task 3 — Balanced accuracy on the development-period results
Balanced accuracy is the mean of sensitivity and specificity, so a constant predictor scores exactly 0.50 regardless of class imbalance.

In [ ]:
bench_rows = []

for ticker in TICKERS:
    for period, df in [("development", dev[ticker]), ("holdout", hold[ticker])]:
        y = df["target_direction"].values
        bench_rows.append(score_all(y, np.ones(len(y), dtype=int),
                                    ticker=ticker, period=period,
                                    model="majority_class"))
        bench_rows.append(score_all(y, (df["ret_lag_1"].values > 0).astype(int),
                                    ticker=ticker, period=period,
                                    model="persistence"))

benchmarks_bal = pd.DataFrame(bench_rows)

print("Benchmarks with balanced accuracy\n")
print(benchmarks_bal.pivot_table(index=["period", "model"],
                                 values=["accuracy", "balanced_acc", "mcc"])
                    .round(4).to_string())

print("\n\nPer asset, holdout period\n")
print(benchmarks_bal.query("period == 'holdout'")
      [["ticker", "model", "accuracy", "balanced_acc", "mcc"]]
      .to_string(index=False))

print("\nMajority class scores 0.50 balanced accuracy and 0.0000 MCC by")
print("construction, while its raw accuracy reflects only class imbalance.")

benchmarks_bal.to_csv(f"{RESULTS}/benchmarks_balanced{suffix}.csv", index=False)

## 6. Append to the results log

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

L = []
L.append(""); L.append("---"); L.append("")
L.append(f"## {today} - Notebook 09: supervisor-requested analyses"
         f"{' (QUICK TEST)' if QUICK_TEST else ''}")
L.append("")
L.append("**Task 1: neural models on the 2024-2025 holdout**")
L.append("Trained on all development data to 2023-12-31; scaler fitted on")
L.append("development only; holdout used once with no subsequent tuning.")
L.append("")
L.append(summary_holdout.to_string())
L.append("")
L.append("**Task 2: scaling factor isolated for neural models**")
L.append("Walk-forward split and direction target in both arms; only the")
L.append("scaler's fitting window differs.")
L.append("")
L.append(eff.to_string())
L.append("")
L.append(f"- Pooled effect: {overall:+.3f} percentage points across {len(pv)} matched pairs")
L.append(f"- Pairs with non-zero difference: "
         f"{int((pv['difference'].abs() > 1e-9).sum())}/{len(pv)}")
L.append("- Ordinary least squares comparison: exactly 0.000, by equivariance")
L.append("  of OLS to affine rescaling. The argument does not extend to")
L.append("  gradient-descent training, which is why this was tested directly.")
L.append("")
L.append("**Task 3: balanced accuracy**")
L.append("")
L.append(benchmarks_bal.pivot_table(index=["period", "model"],
                                    values=["accuracy", "balanced_acc", "mcc"])
                       .round(4).to_string())

entry = "\n".join(L)
with open(f"{BASE}/results_log.md", "a") as f:
    f.write(entry)
print(entry)